# Кластеризация текстов

# 0. Сначала уточнить задачу

Кластеризация группирует похожие тексты без готовых меток. До выбора метода стоит выяснить:

- Что именно хотим группировать: отзывы, обращения, документы? Что значит «похожие»: об одной теме, с одинаковым намерением или с похожей формулировкой?
- Нужно ли заранее знать число групп?
- Что делать с короткими, шумными и смешанными по смыслу текстами?

| Цель | Что должно быть близко | Подход |
|---|---|---|
| Темы / рубрикация («спорт», «политика») | общая тематика | TF-IDF, эмбеддинги + KMeans, BERTopic |
| Интенты обращений в поддержку | смысл запроса, а не слова | sentence-эмбеддинги + HDBSCAN / граф |
| Дедупликация, near-duplicates | почти идентичный текст | MinHash/SimHash или эмбеддинги + порог косинуса |
| Стиль/авторство | лексика, синтаксис | символьные n-граммы, стилометрия |

### BERTopic — это модульный конвейер из 5 шагов

Документы
  → 1. Embeddings (sentence-transformers)   — семантические векторы
  → 2. UMAP                                  — сжатие размерности
  → 3. HDBSCAN                               — кластеризация (= темы)
  → 4. CountVectorizer + c-TF-IDF            — ключевые слова каждой темы
  → 5. (опционально) улучшение представления — MMR, KeyBERT, LLM-названия

### нюансы

Для документов обычно сравнивают направление TF‑IDF-векторов, а не их абсолютную величину. Поэтому важно различать сами метрики и то, как документы нормализованы.

- Евклидово расстояние измеряет прямую дистанцию между векторами. Без нормализации оно зависит и от тематики, и от длины документа: длинные тексты часто получают большие TF‑IDF-векторы и могут оказаться далеко от коротких даже при похожей лексике.

- Косинусное расстояние обычно определяют как 1 − cosine similarity. Оно сравнивает направление векторов и почти не учитывает их масштаб. Для текстов это часто полезно: документы могут быть похожи по терминам, хотя один заметно длиннее другого.

Что меняет L2-нормализация: 

- для попарного сравнения евклидово и косинусное расстояния задают одинаковый порядок похожести.

- обычный k-means минимизирует сумму квадратов евклидовых расстояний до центроидов. Даже если документы нормированы, центроиды в ходе алгоритма обычно не имеют длины 1. Поэтому обычный k-means по L2-нормированному TF‑IDF не полностью эквивалентен кластеризации по косинусному расстоянию


Внимание на формулs! Реализация tf-idf в sklearn

tfidf(t,d) = count(t,d) × ( ln((1+N)/(1+df(t))) + 1 )


# Данные

## 1. Подготовка данных

1. 
- проверьте язык и кодировку.
- Удалить точные дубликаты и почти-дубликаты (иначе они образуют вырожденные кластеры и тянут центроиды).
- Отфильтровать мусор: пустые,, служебные шаблоны, подписи в письмах, HTML, лишние пробелы.

2. 
- Нормализовать текст настолько, насколько нужно задаче. Для TF-IDF: lowercasing, лемматизация (для русского — `pymorphy3` / `spaCy ru_core_news_sm` / `Natasha`), стоп-слова.
- Для нейросетевых эмбеддингов: **агрессивная чистка вредна** — модели обучены на естественном тексте. Достаточно убрать HTML и явный мусор. Сравнивайте варианты на примерах.

3. 
- слишком короткие (< 3–5 слов): очистка может убить весь смысл
- Длинные документы: разбить на чанки (по 200–500 токенов) и усреднить векторы, либо взять первый абзац + заголовок. Учтите лимит модели (обычно 512 токенов).

4.

Лемматизация vs стемминг — любят спрашивать: 
- стемминг = обрезать суффиксы по правилам (быстро, грубо, «data→dat»); 
- лемматизация = словарная форма слова с учётом части речи (медленнее, чище, «better→good»).

| Операция | Зачем | Библиотека |
|---|---|---|
| нижний регистр | «Кофе» = «кофе» | `.lower()` |
| удаление пунктуации/HTML | шум не влияет на смысл | `re`, регулярки |
| токенизация | разбить на слова | `spaCy`, `nltk`, `re.split` |
| лемматизация | «бегущий/бегу/бежал» → `бег` | `spaCy` (мorfologer) |
| стемминг | грубее лемматизации: «running»→`run` | `nltk.PorterStemmer` |
| удаление стоп-слов | «и, в, на» не несут смысла | зависит от задачи! |

## 2. Векторизация

**Baseline (обязательно сделайте его):** TF-IDF + `TruncatedSVD` (LSA (Latent Semantic Analysis) = «взять TF-IDF матрицу и сжать её через SVD». TruncatedSVD — инструмент sklearn, который это делает.) до 100–300 компонент. Быстро, интерпретируемо, часто достаточно для тематической рубрикации.

**Основной современный вариант:** sentence-эмбеддинги.
- Мультиязычные / русские: `intfloat/multilingual-e5-base` (или `large`), `BAAI/bge-m3`, `ai-forever/sbert_large_nlu_ru`, `paraphrase-multilingual-mpnet-base-v2`.
- `normalize_embeddings=True` — тогда евклидово расстояние монотонно связано с косинусным, и KMeans корректно работает «по косинусу».

**Гибрид:** конкатенация нормированных TF-IDF-SVD и эмбеддингов иногда лучше ловит и лексику, и семантику. Проверяйте на данных.

### 2.1. Bag-of-Words и TF-IDF классика

Bag-of-Words Идея: документ = «мешок слов». Порядок теряется, остаётся счёт, какие слова и как часто.

TF-IDF — хороший простой и объяснимый базовый вариант. Он выделяет слова, важные для документа и нечастые во всём корпусе. Подходит для тематически различающихся текстов, особенно если важны конкретные термины. TF-IDF — хороший простой baseline. Подходит, когда важны конкретные термины и тематика корпуса. Минус: слова с близким смыслом, но разным написанием, обычно оказываются далеко друг от друга.

Полученную таблицу 'слова-документы' можно рассматривать и как векторизацию слов и как векторизацию документов. В обычной матрице TF-IDF строка — вектор документа, столбец — веса одного термина по документам. Столбцы сами по себе не являются обученными семантическими векторами слов.

### 2.2. Word2Vec (вектор на слово) 

Важно: похожие слова не всегда означают похожий смысл — и наоборот. Косинусная близость эмбеддингов ловит **семантику**, а не темы: «купить айфон» и «продать айфон» будут рядом. Если это не то, что вам нужно — закладывайте это в решение заранее.

Word2Vec даёт статический вектор слова. Свойство, которое спрашивают как маркер понимания: vec("король") − vec("мужчина") + vec("женщина") ≈ vec("королева") — векторы ловят смысловые отношения как арифметику.

Идея: слово описывается контекстом вокруг себя. Обучаем нейросеть предсказывать соседние слова — и её внутренний слой даёт dense-вектор на каждое слово.
- Skip-gram — по слову предсказываем контекст (лучше для редких слов);
- CBOW - Continuous Bag-of-Words — по контексту предсказываем слово (быстрее, чаще).

Обе архитектуры — мелкие нейросети с одним скрытым слоем без нелинейности, решающие фиктивную задачу: предсказывать слова по их соседям в окне. Предсказания никому не нужны — настоящий продукт это веса, которые и становятся эмбеддингами.

| Модель |Входной слой| Cкрытый слой| Выходной слой|
|---|---|---|---|
|Skip-gram| одно целевое слово в виде one-hot вектора размерности V|Вектор входного слова умножается на весовую матрицу M, и мы получаем его эмбеддинг h размерности Q|Вектор h используется для предсказания каждого слова из контекста по отдельности. Для каждого предсказания применяется Softmax, и модель обучается максимизировать вероятность появления правильных контекстных слов.Для каждого слова модель пытается «размножить» его вектор, чтобы он был хорошим предсказателем для всех слов, которые могут встретиться рядом с ним.|
|CBOW|несколько контекстных слов. Каждое слово представлено в виде one-hot вектора размерности V (размер словаря). Порядок слов в контексте не важен |Векторы всех контекстных слов умножаются на общую весовую матрицу M (размером V x Q, где Q — желаемая размерность эмбеддинга). Затем полученные векторы усредняются, образуя один вектор контекста h размерности Q| Вектор h умножается на другую весовую матрицу M' (размером Q x V) и пропускается через функцию Softmax. Результат — распределение вероятностей по всему словарю. Модель обучается так, чтобы вероятность правильного целевого слова была максимальной.|

Вычисление полного Softmax по всему словарю (особенно если он содержит миллионы слов) — очень дорогая операция. Поэтому в Word2Vec используются две хитрости:

    Negative Sampling (Негативное сэмплирование): Вместо того чтобы предсказывать вероятность для всех слов, модель учится решать более простую задачу: отличать «настоящие» пары (целевое слово — слово из его реального контекста) от «поддельных» пар (целевое слово — случайное слово, не встречающееся в этом контексте). Это радикально ускоряет обучение.

    Subsampling (Субдискретизация): Слишком частые слова (например, «в», «и», «на») не несут много смысла, но создают огромное количество обучающих примеров. Поэтому при обучении некоторые частые слова случайно «выбрасываются» из текста с определенной вероятностью, чтобы модель фокусировалась на более информативных редких словах.
    
Но word2vec даёт один вектор на слово, не на текст. Как получить документ? Среднее по словам (mean of word vectors) — грубо и теряет порядок слов. Отсюда — эмбеддинги.


### 2.3. Документные / контекстные эмбеддинги (TRANSFORMERS, sentence-transformers)

Эмбеддинги — плотные векторы, которые отражают семантическое сходство. Они полезны, когда тексты говорят об одном разными словами. Для выбора embedding-модели учитывайте язык, длину текстов и тип задачи. sentence-эмбеддинги — вектор целого текста, учитывающий контекст и порядок слов

Идея: взять BERT-подобную модель, обученную на masked language model, и получить один dense-вектор на весь текст так, что смысл (порядок, контекст) учтён.

### 2.4. БОНУС дообучение современного декодера как энкодера


### Резюме

- нужен максимально быстро и прозрачно, мало данных, классический ML → TF-IDF; 

- нужен смысл/синонимы/контекст, данных достаточно, можно тянуть модель → transformer-эмбеддинги.

На собеседовании про AdTech и «развивать инфраструктуру» правильный ответ — зависит от задачи и ресурсов, а не «BERT лучше всегда». Скажи: «TF-IDF как бейзлайн и быстрый baseline, эмбеддинги — когда ловим семантику».

## 3. Снижение размерности

Не обязательно, но часто полезно, например, для визуализации. Может исказить расстояния и структуру исходных embeddings:  оценивайте устойчивость и содержательность кластеров, поскольку он не сохраняет плотность и глобальные расстояния идеально.

- `TruncatedSVD` / PCA → 50–100 компонент: линейно, быстро, безопасно.
- `UMAP` → 5–15 компонент, `metric="cosine"`, `min_dist=0.0`: сильно улучшает разделимость, но **искажает расстояния и создаёт ложные структуры**. Ставьте `random_state` для воспроизводимости.
- t-sne: медленнее umap и почти всегда уступает


**Зачем.** TF-IDF-матрица имеет размерность = словарь (сотни тысяч колонк). Косинусная близость в стольких измерениях становится разреженной и неровной («проклятие размерности»): все точки почти одинаково далеко. Понижаем до 20–100 измерений → кластеры ищутся устойчиво.

| Метод | Тип | Когда |
|---|---|---|
| **SVD / TruncatedSVD (=LSA)** | линейный | для **разреженной** TF-IDF-матрицы; ловит скрытые семантические темы |
| **PCA** | линейный | только для **dense** матриц; максимизирует дисперсию |
| **UMAP / t-SNE** | нелинейный | для плотных эмбеддингов и **визуализации** (UMAP ещё и для кластеринга перед DBSCAN) |

# Модель

## 4. Выбрать алгоритм кластеризации

- K-Means — если примерно знаете число кластеров и хотите простой отправной пункт. Нужно задать k; может плохо работать с кластерами разной формы и размера. «`k` понятен из бизнеса, много данных, нужна скорость» → KMeans на L2-нормализованных векторах;
    
- Иерархическая кластеризация — если полезно смотреть на группы разных уровней детализации. Может быть затратной на больших наборах. «хочу посмотреть иерархию тем» → Agglomerative.

- HDBSCAN — если число кластеров заранее неизвестно, а часть текстов может оказаться шумом. Умеет не присваивать некоторым текстам кластер, но результат зависит от параметров и плотности данных. «не знаю `k`, тексты короткие, хочу отбросить мусор» → HDBSCAN/DBSCAN (шум);

Для короткого базового эксперимента часто берут TF-IDF + KMeans. Если число групп заранее неизвестно, можно проверить плотностный алгоритм на эмбеддингах. Практическое правило: **KMeans как baseline → HDBSCAN как основная гипотеза → сравнить**.

### KMeans: как устроен 

Инициализировали `k` центроидов → итеративно 
1. каждая точка ставится в соответствие ближайшему центроиду, 
2. центроиды пересчитываются в среднее облака — покаAssignment перестанет меняться. 

Выбор `k` для KMeans (вот где реально спрашивают):

У KMeans нет встроенного способа определить «правильное» число кластеров: разные метрики оценивают разные свойства разбиения. Поэтому обычно проверяют несколько значений k и сопоставляют качество, устойчивость и смысл кластеров для задачи.

1. Метод локтя (elbow) — инерция.
Для каждого k запускают KMeans и считают инерцию: сумму квадратов расстояний объектов до центроидов своих кластеров:
$$inertia=∑_{i=1}^n∥x_i−c_i∥$$
При увеличении k инерция всегда уменьшается: большее число центроидов позволяет точнее приблизить данные. Ищут значение, после которого снижение становится заметно медленнее — условный «излом» графика, либо выход на плато. Минус: «излом» часто неточный или его нет.
2. Силуэт (silhouette score) — насколько кластеры разделены.
Для каждого объекта сравнивают его близость к своему кластеру с близостью к ближайшему соседнему кластеру. Средний силуэт обычно лежит от −1 до 1: чем он выше, тем лучше объекты вписаны в свои кластеры и отделены от других. Можно сравнить средний силуэт для разных k и искать высокое значение.
Плюс: метрика оценивает не только компактность кластеров, но и их разделённость. Минус: она может предпочитать слишком простое разбиение и не всегда хорошо отражает качество кластеров разного размера или плотности.
3. **Стабильность**: бутстрэп/подвыборки 80% данных, кластеризуем, считаем ARI между запусками. Устойчивое решение важнее красивого силуэта.

Не гонитесь за «правильным k» — его обычно не существует. Ориентируйтесь на пригодность для задачи.

**Иерархия**

Иерархическая кластеризация строит дерево вложенных групп. В отличие от KMeans, алгоритм обычно не требует заранее задавать число кластеров: он сначала строит иерархию, а число групп выбирают позже — например, «разрезая» дерево на нужной высоте.

Чаще всего используют агломеративный подход:

- Каждая точка изначально образует отдельный кластер.
- На каждом шаге алгоритм объединяет две наиболее близкие группы.
- Объединения продолжаются, пока все точки не окажутся в одном дереве.

Реже применяют дивизивный подход: начинают с одного общего кластера и последовательно делят его на части.

ВАЖНО:
- Масштабирование признаков важно. Если один признак измеряется тысячами, а другой долями единицы, первый может почти полностью определять расстояние. Обычно числовые признаки предварительно масштабируют.
- Расстояние Уорда между кластерами — не универсальный выбор. Он предназначен для евклидовой геометрии и ориентирован на компактные кластеры. Его не следует бездумно сочетать с произвольной метрикой.
- Высота на дендрограмме — не готовая оценка полезности кластера. Это не вероятность принадлежности или мера «истинности» группы.

**HDBSCAN**
HDBSCAN — алгоритм кластеризации по плотности. В отличие от KMeans, ему не нужно заранее задавать число кластеров. Он может выделять кластеры разной формы, находить некоторые кластеры с разной плотностью и помечать неоднозначные или изолированные точки как шум. min_cluster_size	Минимальный размер группы, которую разрешено считать кластером	Большие значения отбрасывают мелкие группы; маленькие могут создавать много небольших кластеров min_samples	Насколько строгой должна быть оценка плотности.

Упрощённо алгоритм работает так:

- Оценивает локальную плотность вокруг каждой точки по расстояниям до её ближайших соседей. Пусть d(x,y) — расстояние между точками, а k=min_samples. Core distance точки x — расстояние до её k-го ближайшего соседа:
core⁡k(x)=d(x,  k-й ближайший сосед x). Чем больше это расстояние, тем ниже локальная плотность вокруг точки. У разных реализаций есть небольшие различия в том, включается ли сама точка в подсчёт соседей.

- Строит иерархию кластеров: на разных уровнях плотности точки объединяются в более крупные или разделяются на более мелкие группы. HDBSCAN заменяет обычное расстояние на mutual reachability distance: dmreach,k(a,b)=max⁡{core⁡k(a),core⁡k(b),d(a,b)}. По взаимным расстояниям строят граф и по рёбрам образуют кластеры: если оставить только рёбра с весом не больше ε, кластеры = связные компоненты графа  на этом уровне.

- Оценивает, какие кластеры сохраняются устойчивыми на широком диапазоне плотностей, и выбирает их.

Поэтому HDBSCAN ищет не просто группы близких точек, а группы, которые выглядят достаточно устойчивыми в этой иерархии. Точки, не вошедшие в выбранные кластеры, получают метку шума — обычно -1.


**LDA**
Идея LDA:  документ задаёт, какие темы в нём содержатся, тема задаёт, какие слова в ней характерны, а LDA пытается восстановить эти скрытые распределения по текстам. Например, документ может на 70% относиться к теме «спорт» и на 30% — к теме «политика»; тема «спорт» при этом может часто включать слова «команда», «матч» и «игрок».

 Для документа d это вектор $θ_d$, где $θ_{dk}$ — доля темы k, а все доли суммируются до 1:
$∑_{k=1}^K θ_{dk}=1$

У каждой темы тоже есть распределение: $ϕ_{kv}$ — вероятность слова v в теме k, причём
$∑_{v=1}^V ϕ_{kv}=1$

Главные гиперпараметры модели

    K — число тем. Его задают до обучения. LDA оценивает содержание этих тем, но сама обычно не выбирает их количество.
    α — prior для распределения тем в документах. Малое αα обычно поощряет документы с несколькими доминирующими темами; большое — документы, где доли тем более равномерны.
    β — prior для распределения слов в темах.
    
LDA работает с Bag-of-Words: учитывает, какие слова встречаются и сколько раз, но обычно игнорирует их порядок.

| Алгоритм | Как определяет «кластер» | Плюс | Минус | Нужен ли `k` |Когда брать |
|---|---|---|---|---|---|
| **KMeans** | сферическое облако вокруг центроидов | быстрый, понятный, масштабируется | форма обязана быть сферической; чувствителен к инициализации, нужно задавать `k`, все точки распределены, любит шарообразные кластеры | **да** | известно примерное `k`, кластеры сопоставимы по размеру, нужен быстрый baseline и предсказание для новых текстов |
| **MiniBatchKMeans** | то же, на подвыборках | для огромных датасетов | чуть хуже качество | да | |
| **DBSCAN** | плотность + `eps` | находит произвольные формы; **шум как отдельный класс `-1`** | чувствителен к `eps`, на текстах плохо при большой размерности | нет (но нужны 2 порога) ||
| **HDBSCAN** | иерархия плотностей | устойчивее к параметрам | нужен UMAP, много точек уходит в `-1`, нет `predict` для новых данных | нет |`k` неизвестен, кластеры разной плотности и размера, есть шум/выбросы|
| **Agglomerative (иерархическая)** | сливает ближайшие кластеры | даёт дендрограмму, не нужен `k` | медленная на больших данных O(n²) по памяти | нет (выбираем отрезая дендрограмму) |иерархия/таксономия, можно резать по порогу расстояния вместо `k`|


| Алгоритм | Когда брать | Минусы |
|---|---|---|
| **Граф: kNN + Leiden/Louvain** | интенты, дедупликация, большие корпуса | подбор порога相似度, требует faiss |
| **BERTopic** | нужен готовый «эмбеддинги → UMAP → HDBSCAN → c-TF-IDF → темы» | обёртка, меньше контроля |
| **GMM / LDA** | мягкое распределение по темам, документ = смесь тем | LDA хуже на коротких текстах |

# Метрики

## 5. Проверить качество — и смысл, и числа

Главное, что надо проговорить словами: **у кластеризации нет правильного ответа** — ground truth нет, поэтому и метрики «относительные». Это и есть ответ на вопрос «как выбрать метрику под бизнес-задачу». Только метрика не доказывает, что кластеры полезны.

### Если есть хотя бы небольшая разметка (200–500 размеченных примеров сильно окупаются): ARI, NMI, V-measure, purity кластеров.

!!! Можно разметить LLM

- **Adjusted Rand Index (ARI)** — попарная правильность разделения, −1…1, независим от переметки.
- **Normalized Mutual Information (NMI)** — «сколько информации кластеры несут о правде», 0…1.
- **Homogeneity / Completeness / V-measure** — «кластер не смешал темы?» / «одна тема не рассыпалась по кластерам?».

### Если разметки нет

**Silhouette** ∈ [−1, 1]: 1 = «в своём кластере и далеко от чужих». Смотрят не столько абсолют, сколько «при каком `k`/`eps` максимум».

**Davies–Bouldin**: ближе к 0 — лучше (средняя «близость соседних кластеров»).

**Inertia / distortion** (для локтя KMeans).

- Silhouette / Davies–Bouldin / Calinski–Harabasz — Они помогают сравнивать варианты, но высокий балл не гарантирует полезные для бизнеса кластеры. ⚠️ Только для *сравнения конфигураций на одних и тех же векторах*. Сравнивать силуэт TF-IDF и эмбеддингов бессмысленно. ⚠️ Силуэт, посчитанный в UMAP-пространстве, завышен — UMAP сам создаёт разрывы. Считайте на исходных векторах. Silhouette score может помочь сравнить варианты, но для текстов в пространствах высокой размерности не всегда хорошо отражает человеческое восприятие.

- Доля шума в HDBSCAN (>40–50% — тревожный знак).

- Размеры кластеров: один кластер на 70% корпуса = модель не разделила.

### BEST PRACTICE

**Проверка "руками"**
- **Ручная проверка**: Для каждого кластера просмотрите типичные тексты и тексты около границ кластеров. по 10 случайных текстов из каждого кластера + 5 самых дальних от центроида. Дайте кластерам короткие описания и проверьте, можно ли их интерпретировать. Это до сих пор самый надёжный метод. для каждого кластера — топ слов (по TF-IDF-весу) или representative-документы (ближайшие к центроиду). Кластер «красивый по метрике», где топ-слова не складываются в тему, — мусор.
- **LLM-as-judge**: дайте модели 15 текстов кластера и попросите оценить связность по шкале 1–5 и сформулировать общий признак. Дёшево и масштабируемо.

**Проверьте устойчивость** 
- сохраняются ли группы при небольших изменениях параметров или выборки?

**Практическая проверка. Интерпретация кластеров** 
- ответить, помогают ли группы решить задачу — например, найти темы обращений или объединить похожие отзывы.
- **c-TF-IDF** (как в BERTopic): treats кластер как один большой документ, достаёт слова, отличающие его от остальных.
- Топ-термы по TF-IDF внутри кластера минус топ-термы корпуса.

# 6. Частые ошибки

**Общая культура.** 

- Сравнивать алгоритмы на разных предобработках, не фиксируя остальные условия.

- Смешаны языки без мультиязычной модели. **Тексты на русском, лемматизация из английского словаря** — слова искажаются. Нужен `pymorphy2`/`Natasha`. 

- путаница между метриками косинуса и евклидовой. KMeans по TF-IDF без нормализации → кластеры по длине текста. Евклидово расстояние по умолчанию не всегда осмысленно для категориальных признаков, текстов, эмбеддингов или смешанных типов данных. Кодирование признаков само по себе не гарантирует, что расстояния после него подходят для кластеризации.

- Не убрать дубликаты и шаблонные фразы — из-за этого алгоритм группирует форматы, а не смысл. Один шаблонный текст может искусственно сформировать группу. 

- Не масштабировать признаки. Если один признак измеряется тысячами, а другой долями единицы, первый может почти полностью определять расстояния. Обычно перед кластеризацией признаки масштабируют — если выбранная метрика и природа данных не требуют другого подхода.

**Шум, короткие и длинные тексты**

- Не проверять, что именно попало в кластер, включая выбросы и короткие тексты. Не проверить шум, редкие темы и смешанные сообщения. 

- Очень короткие тексты (поисковые запросы, 2–4 слова) через TF-IDF → почти нулевые пересечения словарей; тут только эмбеддинги (+ иногда конкатенация с контекстом/сессией). **Синтетический сигнал от стоп-слов** — после удаления может стать **хуже** на коротких текстах; проверяй ablation. **`n_gram=2` на коротких текстах** — признаки разрежаются, модель голодает.

**Оценка без просмотра глазами.**

- Считать высокий silhouette гарантией полезных групп. Считать, что каждый кластер обязательно соответствует реальной теме.

- Выбрать число кластеров без проверки их интерпретируемости. Выбирать количество кластеров только по одной метрике. 

- Ожидание, что кластеры совпадут с «естественными» категориями бизнеса. Обычно нужна последующая ручная сборка/переименование кластеров в таксономию. Принять тематические кластеры за категории намерений пользователя.

- Не делайте вывод по красивой 2D-картинке: визуализация помогает исследовать данные, но сама по себе не подтверждает качество кластеризации.

**Проклятье размерности**

- Silhouette по всей матрице при N=10⁵ → память/время O(N²); считай по подвыборке.

- HDBSCAN прямо в 768-мерном пространстве → почти всё уходит в шум.

- С другой стороны, снижение размерности может изменить расстояния и локальную плотность, а значит и кластеры. Полезно, но результаты нужно проверять на исходных признаках или независимыми способами.